# Oracle LoRA fine-tune — Colab Pro (A100 80GB)

Fine-tunes `DeepSeek-R1-Distill-Qwen-7B` into an agentic Oracle model using your own Claude Code session traces as SFT corpus.

**Runtime:** `Runtime → Change runtime type → A100` (V100 / L4 also work, just slower).

**Walltime:** ~3–5 h on A100 80GB, 6–8 h on V100 16GB (QLoRA kicks in on the smaller card).

**Corpus transport:** Colab joins your Tailscale tailnet and SFTPs the corpus from the DO droplet (`100.67.227.31`) — nothing private goes through Drive.

## 1. GPU sanity check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Mount Google Drive (outputs only)

Durable storage for the LoRA adapter + final GGUF. The raw corpus is **not** here — it comes from your SSH host in cell 3b.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUTPUT_DIR = '/content/drive/MyDrive/oracle-lora/out'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'outputs → {OUTPUT_DIR}')

## 3a. Join the Tailscale tailnet

The DO droplet's public SSH port is firewalled to Tailscale-only, so Colab must join the tailnet first. Tailscale's `--tun=userspace-networking` mode + SOCKS5 proxy is the right config for a container without `CAP_NET_ADMIN` (which Colab doesn't grant).

Generate a **reusable, ephemeral 24 h** auth key at <https://login.tailscale.com/admin/settings/keys> and paste it into the prompt.

In [ ]:
import getpass, time, subprocess
TS_AUTHKEY = getpass.getpass('Tailscale auth key (tskey-auth-...): ')

!curl -fsSL https://tailscale.com/install.sh | sh

# Start tailscaled in userspace-networking mode with a SOCKS5 proxy on 1055.
# Regular TUN mode fails in Colab (no CAP_NET_ADMIN); userspace works everywhere.
subprocess.Popen(
    ['sudo', 'tailscaled', '--tun=userspace-networking',
     '--socks5-server=localhost:1055', '--state=mem:'],
    stdout=open('/tmp/tailscaled.log', 'w'), stderr=subprocess.STDOUT,
)
time.sleep(3)

!sudo tailscale up --authkey={TS_AUTHKEY} --hostname=colab-oracle-sft --accept-dns=false
!tailscale status | head -20

## 3b. Pull corpus via SFTP (paramiko over Tailscale SOCKS5)

Paramiko doesn't speak SOCKS5 natively, so we monkey-patch Python's stdlib `socket` with `PySocks` — every TCP connect after this point transparently goes through `localhost:1055`, which is the Tailscale userspace proxy into the tailnet.

The tarball is already staged at `devops@100.67.227.31:/tmp/oracle-lora.tar.gz`.

In [ ]:
# --- edit if needed ---
HOSTNAME    = '100.67.227.31'            # DO droplet (AutomataNexus) Tailscale IP
USERNAME    = 'devops'
PORT        = 22
REMOTE_PATH = '/tmp/oracle-lora.tar.gz'
LOCAL_PATH  = '/content/oracle-lora.tar.gz'
# -----------------------

!pip install -q paramiko pysocks

import socks, socket, paramiko, getpass, os

# Route all TCP through Tailscale's userspace SOCKS5 proxy.
socks.set_default_proxy(socks.SOCKS5, 'localhost', 1055)
socket.socket = socks.socksocket

password = getpass.getpass(f'{USERNAME}@{HOSTNAME} password (blank for key-auth): ')
key_path = '' if password else input('Path to SSH private key: ').strip()

transport = paramiko.Transport((HOSTNAME, PORT))
if password:
    transport.connect(username=USERNAME, password=password)
else:
    pk = paramiko.RSAKey.from_private_key_file(os.path.expanduser(key_path))
    transport.connect(username=USERNAME, pkey=pk)

sftp = paramiko.SFTPClient.from_transport(transport)
print(f'pulling {REMOTE_PATH}...')
sftp.get(REMOTE_PATH, LOCAL_PATH)
sftp.close(); transport.close()
print(f'got {LOCAL_PATH}: {os.path.getsize(LOCAL_PATH)/1e6:.1f} MB')

## 4. Extract corpus

Extracts to `/content/oracle-lora/` — fast local SSD.

In [ ]:
!rm -rf /content/oracle-lora
!tar xzf {LOCAL_PATH} -C /content/
!ls -lh /content/oracle-lora/
!cat /content/oracle-lora/corpus_stats.json

## 5. Install training dependencies

Unsloth gives 2–3× throughput over bare `transformers + peft` via custom Triton kernels for RoPE/RMSNorm/SwiGLU.

In [ ]:
# Restore normal DNS/socket — we no longer need SOCKS5 for pip mirrors.
import socket as _s, socks
_s.socket = socks.socksocket.__mro__[1]  # reset to built-in socket
!pip install -q -U unsloth bitsandbytes peft trl accelerate datasets

## 6. Load base model + attach LoRA

- **rank=16** — style + tool-convention shift, not new domain.
- **all 7 linear projections** — standard target set for Qwen/DeepSeek.
- **4-bit base + bf16 compute** — fits A100 40GB, L4 24GB comfortably.

In [ ]:
from unsloth import FastLanguageModel
MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='deepseek-ai/DeepSeek-R1-Distill-Qwen-7B',
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto: bf16 on A100/L4, fp16 on V100
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print(model.print_trainable_parameters())

## 7. Load SFT corpus

Each line is `{"text": "<DeepSeek-R1-formatted session>"}` — template already baked in (full-width-pipe `<｜User｜>`/`<｜Assistant｜>` + inline `<tool_use>{json}</tool_use>` blocks matching nexus-serve's parser).

In [ ]:
from datasets import load_dataset
train_ds = load_dataset('json', data_files='/content/oracle-lora/corpus_train.jsonl', split='train')
val_ds   = load_dataset('json', data_files='/content/oracle-lora/corpus_val.jsonl',   split='train')
print(f'train: {len(train_ds)}   val: {len(val_ds)}')
print('sample head:', train_ds[0]['text'][:400])

## 8. Train

Checkpoints every 200 steps to Drive so a disconnect costs at most ~5 min of work.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=True,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        weight_decay=0.01,
        bf16=True,
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=100,
        save_strategy='steps',
        save_steps=200,
        save_total_limit=3,
        report_to='none',
        seed=42,
    ),
)
trainer.train()

## 9. Save final adapter to Drive

In [ ]:
FINAL_DIR = f'{OUTPUT_DIR}/final'
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
!ls -lh {FINAL_DIR}

## 10. Merge LoRA → base (fp16)

In [ ]:
import gc, torch
del model, trainer
gc.collect(); torch.cuda.empty_cache()

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=FINAL_DIR,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
)
MERGED_DIR = '/content/oracle-merged'
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
!ls -lh {MERGED_DIR}

## 11. Convert merged fp16 → GGUF Q4_K_M

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON
!cmake --build /content/llama.cpp/build -j --target llama-quantize llama-cli

In [ ]:
GGUF_FP16 = '/content/oracle-fp16.gguf'
GGUF_Q4KM = f'{OUTPUT_DIR}/oracle-r1-distill-q4km.gguf'

!python /content/llama.cpp/convert_hf_to_gguf.py \
    {MERGED_DIR} --outfile {GGUF_FP16} --outtype f16

!/content/llama.cpp/build/bin/llama-quantize {GGUF_FP16} {GGUF_Q4KM} Q4_K_M

!ls -lh {GGUF_Q4KM}

## 12. Sanity-check the GGUF (optional)

In [ ]:
!/content/llama.cpp/build/bin/llama-cli \
    -m {GGUF_Q4KM} \
    -p "<｜begin▁of▁sentence｜>You are Oracle.<｜User｜>What is 2+2?<｜Assistant｜>" \
    -n 32 --temp 0

## 13. Push the GGUF back over SFTP

Re-routes through Tailscale SOCKS5 (same proxy is still up from cell 3a), pushes to `/tmp/` on the droplet.

In [ ]:
PUSH_REMOTE_PATH = '/tmp/oracle-r1-distill-q4km.gguf'

import socks, socket, paramiko, getpass, os
socks.set_default_proxy(socks.SOCKS5, 'localhost', 1055)
socket.socket = socks.socksocket

password = getpass.getpass(f'{USERNAME}@{HOSTNAME} password (blank for key-auth): ')
key_path = '' if password else input('Path to SSH private key: ').strip()

transport = paramiko.Transport((HOSTNAME, PORT))
if password:
    transport.connect(username=USERNAME, password=password)
else:
    pk = paramiko.RSAKey.from_private_key_file(os.path.expanduser(key_path))
    transport.connect(username=USERNAME, pkey=pk)

sftp = paramiko.SFTPClient.from_transport(transport)
print(f'pushing {GGUF_Q4KM} → {HOSTNAME}:{PUSH_REMOTE_PATH}...')
sftp.put(GGUF_Q4KM, PUSH_REMOTE_PATH)
sftp.close(); transport.close()
print(f'done. On the local box, run:')
print(f'  scp {USERNAME}@{HOSTNAME}:{PUSH_REMOTE_PATH} /opt/AxonML/models/oracle-distill/')

## 14. Done — wire into nexus-serve

On the local host:

```bash
mkdir -p /opt/AxonML/models/oracle-distill
scp devops@100.67.227.31:/tmp/oracle-r1-distill-q4km.gguf \
    /opt/AxonML/models/oracle-distill/oracle-r1-distill-q4km.gguf

pkill -f nexus-serve; sleep 2
nohup /opt/AxonML/nexus-serve/target/release/nexus-serve \
  --model /opt/AxonML/models/oracle-distill/oracle-r1-distill-q4km.gguf \
  --port  11436 --quantized > /tmp/nexus_oracle.log 2>&1 &
```

Then flip the NexusOracle header toggle to **Local** and smoke-test per `ORACLE_LORA_FINETUNE.md` § 7.